# Explorer les prix des carburants de 2007 à 2026
Les données proviennent exclusivement des archives officielles de prix-carburants.gouv.fr.
L'année 2026 est partielle. Chaque ligne de prix représente une déclaration.
Les champs géographiques décrivent la station telle qu'elle apparaît dans l'archive annuelle.

Les prix de 2007 à 2021 sont divisés par 1 000. À partir de 2022, ils sont conservés en euros.
Les dates sont préservées sans fuseau ajouté, car les fichiers XML n'en indiquent pas.
Les valeurs invalides sont comptées dans les bilans. Les prix inférieurs à 0,20 €/L ou supérieurs
à 5 €/L sont conservés et signalés par suspect_price. Ce seuil exploratoire est modifiable dans
le script de préparation. Les archives originales permettent de revenir aux données sources.


In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / "data/processed/annual").exists():
    raise FileNotFoundError("Ouvre ce notebook depuis la racine de Fuel_analyser.")
ANNUAL = ROOT / "data/processed/annual"


## Couverture et qualité par année

In [ ]:
reports = [
    json.loads(path.read_text(encoding="utf-8"))
    for path in sorted(ANNUAL.glob("*/quality.json"))
]
quality = pd.DataFrame(reports).set_index("year")
display(quality[[
    "station_count", "retained_rows", "invalid_rows",
    "exact_duplicates_removed", "suspect_price_rows",
    "conflicting_timestamp_rows", "outside_archive_year_rows",
]])


## Choisir une année et un carburant
Change ces deux paramètres puis réexécute les cellules suivantes.

In [ ]:
YEAR = 2022
FUEL = "Gazole"

folder = ANNUAL / str(YEAR)
prices = pd.read_parquet(folder / "prices.parquet", filters=[("fuel", "==", FUEL)])
stations = pd.read_parquet(folder / "stations.parquet")
interruptions = pd.read_parquet(folder / "interruptions.parquet")
display(prices.head())
print(f"{len(prices):,} déclarations de prix pour {FUEL} en {YEAR}.")


## Examiner les données avant l'analyse

In [ ]:
display(prices.dtypes)
display(prices.isna().sum().rename("Valeurs manquantes"))
display(prices["price_eur_l"].describe())
display(prices.loc[prices["suspect_price"]].head(10))
conflicts = prices.duplicated(["station_id", "fuel", "price_reported_at"], keep=False)
print(f"{conflicts.sum():,} lignes présentent plusieurs prix au même instant pour une station.")


## Observer une station
Le tracé montre les prix déclarés d'une station sur l'année choisie.
Les observations atypiques et les instants où plusieurs prix différents sont déclarés
sont écartés de cet exemple graphique. Les interruptions sont affichées pour examen séparé.
Une absence de nouvelle déclaration ne renseigne pas à elle seule la disponibilité.

Pour une courbe nationale, il faudra reconstruire les prix valides par station à des dates
communes, intégrer les interruptions et choisir une règle de fraîcheur. La moyenne directe
des déclarations donnerait plus de poids aux stations qui modifient souvent leurs prix.


In [ ]:
usable = prices.loc[
    ~prices["suspect_price"] & ~prices["outside_archive_year"] & ~conflicts
].copy()
station_id = usable["station_id"].value_counts().index[0]
station_prices = usable.loc[usable["station_id"].eq(station_id)].sort_values("price_reported_at")
display(stations.loc[stations["station_id"].eq(station_id)])
display(interruptions.loc[interruptions["station_id"].eq(station_id)].head(20))

fig, ax = plt.subplots(figsize=(11, 4))
ax.step(station_prices["price_reported_at"], station_prices["price_eur_l"], where="post")
ax.set(title=f"{FUEL} — station {station_id} — {YEAR}", xlabel="Date de déclaration", ylabel="Prix déclaré (€/L)")
ax.grid(alpha=0.2)
fig.tight_layout()
plt.show()


## Tes questions d'analyse
- Quelles années présentent le plus de données manquantes ou atypiques ?
- Comment le nombre de stations suivies évolue-t-il ?
- Observe-t-on des hausses ou baisses rapides pour la station choisie ?
- Les interruptions enregistrées coïncident-elles avec ces périodes ?

Les données de prix permettent de décrire les évolutions observées.
L'attribution à une cause politique ou économique demandera une méthode et des éléments
de contexte documentés.
